In [ ]:
pip install gradio scikit-learn pandas numpy matplotlib

# Learn about the Gradio : https://www.gradio.app/


In [ ]:
import re # regular expressions modules
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB


In [5]:
data = {
    "review": [
        "I loved this movie, it was fantastic!",
        "Worst movie ever, totally boring",
        "Amazing acting and great storyline",
        "I did not like the film at all",
        "Best movie of the year",
        "Terrible plot and bad acting",
        "Absolutely wonderful experience",
        "Waste of time and money",
        "Brilliant direction and screenplay",
        "Horrible movie"
    ],
    "sentiment": [1,0,1,0,1,0,1,0,1,0]
}

df = pd.DataFrame(data)


In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['review'] = df['review'].apply(clean_text)

In [9]:
# Logistic Regression (ML Pipeline)
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("classifier", LogisticRegression())
])


In [8]:
# Naive Bayes Pipeline
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("classifier", MultinomialNB())
])


In [10]:
# train both of the pipline
lr_pipeline.fit(df['review'], df['sentiment'])
nb_pipeline.fit(df['review'], df['sentiment'])

Pipeline(steps=[('tfidf', TfidfVectorizer(stop_words='english')),
                ('classifier', MultinomialNB())])

In [11]:
# Core Prediction
def predict_sentiment(text, model_choice):
    text = clean_text(text)

    model = lr_pipeline if model_choice == "Logistic Regression" else nb_pipeline

    prediction = model.predict([text])[0]
    probabilities = model.predict_proba([text])[0]

    sentiment = "Positive 😊" if prediction == 1 else "Negative 😞"
    confidence = round(np.max(probabilities) * 100, 2)

    # Visualization
    fig, ax = plt.subplots()
    ax.bar(["Negative", "Positive"], probabilities)
    ax.set_ylim(0, 1)
    ax.set_title("Sentiment Probability Distribution")

    return sentiment, f"{confidence}%", fig


In [12]:
interface = gr.Interface(
    fn=predict_sentiment,
    inputs=[
        gr.Textbox(
            lines=4,
            placeholder="Type a movie review, tweet, or feedback..."
        ),
        gr.Radio(
            ["Logistic Regression", "Naive Bayes"],
            label="Choose Model",
            value="Logistic Regression"
        )
    ],
    outputs=[
        gr.Textbox(label="Predicted Sentiment"),
        gr.Textbox(label="Confidence Score"),
        gr.Plot(label="Sentiment Probability")
    ],
    title="Advanced Sentiment Analysis NLP App",
    description="""
    This application performs real-time sentiment analysis using NLP.

    • TF-IDF Vectorization
    • Machine Learning Models
    • Confidence & Probability Visualization
    """,
    theme="default"
)


In [13]:
interface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://deceb707088aa71cd2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
